In [5]:
from pathlib import Path
import shutil
from PIL import Image
import tensorflow as tf

In [6]:
labels_path = Path('../filtered/train/labels_all')
imgs_path = Path('../filtered/train/images_all')
dest_labels_path = Path('../filtered/train/labels')
dest_imgs_path = Path('../filtered/train/images')


In [11]:
if not dest_labels_path.exists() and not dest_imgs_path.exists():
    try:
        dest_labels_path.mkdir(parents=True)
        dest_imgs_path.mkdir()
        print('Path test dataset created!')
    except Exception as e:
        print('Path test dataset creation failed!' , e)

Path test dataset created!


In [12]:
# Sorted out images with only one aircraft and with size less 10% of image size
for file in labels_path.glob('*.txt'):
    
    with open(file, 'r') as f:
        lines = f.readlines()
        
        if len(lines) == 1:
            
            # true for copycopy
            coord = lines[0].strip('\n').split(' ')[1:]
            
            square = float(coord[2]) * float(coord[3])
            
            if square <= 0.015:
              
                img_path = imgs_path/f'{file.stem}.jpg'
                shutil.copy(file,dest_labels_path)
                shutil.copy(img_path,dest_imgs_path)
                
                #true for copy
              

In [13]:
# check if all files are valid
def load_img(path):
      if isinstance(path, Path): 
          path = str(path)
          
      img = tf.io.read_file(path)
      img = tf.io.decode_jpeg(img, channels=3)
      img = tf.cast(img, tf.float32)/255
      img = tf.image.resize(img, [480, 640])
      # img = tf.expand_dims(img, axis=0)
   
      return img

for file in dest_imgs_path.glob('*.jpg'):
    try:
        im = load_img(file)
        
        # check error with yielding wrong shape of img
        if im.shape != (480, 640, 3):
            print(im.shape)
            print(file)
    except:
        print(file, ' - invalid file')
        label_path = dest_labels_path/f'{file.stem}.txt'
        file.unlink()
        label_path.unlink()
    